# Lab 5 (bonus) — From Scratch to LangGraph (§25.3.1)

In Labs 1–4 we wrote every line of the harness ourselves. In this lab we rebuild the **ReAct agent from Lab 1** and the **planning agent from Lab 2** with
**LangGraph**, which models agent execution as a **graph** — a *node* is a computation step, an *edge* is the transition from one step to the next

| LangGraph concept | What we wrote ourselves in Lab 1 |
|---|---|
| **State** — a dict that flows through the graph | the `messages` list |
| **Node** — a function that takes the state and returns the updated parts | `ask_llm`, `run_tool` |
| **Edge** (can be conditional) | `if not calls: return`, `max_iterations` |
| **Checkpointing** | — (none: the framework adds it) |
| the planner → executor → replan graph | `execute_plan()` + `replan()` in Lab 2 |

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §25.3.1 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`lab5-langgraph.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — setup instructions are in Lab 1

The LangGraph version is pinned, because framework APIs change quickly. With a newer version, some lines may need adjusting

In [ ]:
%pip install -q "langgraph==1.2.16" "langchain-openai==1.7.2" "langchain-core==1.7.0" "grandalf==0.8" python-dotenv

In [ ]:
import os, json, operator
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
# ---- from Lab 1: load the API key (Colab Secrets or a .env file) ----
def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

In [ ]:
llm = ChatOpenAI(model="google/gemma-4-26b-a4b-it",
                 base_url="https://openrouter.ai/api/v1", api_key=load_key(),
                 extra_body={"provider": {"require_parameters": True}, "reasoning": {"enabled": False}})

## 1. Tools with `@tool`

We use the same simulated data as Lab 2 (the first search finds only 3 papers) — LangChain's `@tool` builds the schema for us from the **docstring** and **type hints**

In [ ]:
PAPERS = {  # simulated data for teaching (same set as Lab 2)
    "P1": {"title": "Crossformer", "year": 2023, "method": "two-stage attention across dimensions"},
    "P2": {"title": "TimesFM", "year": 2024, "method": "decoder-only foundation model"},
    "P3": {"title": "Chronos", "year": 2024, "method": "tokenizes values for a T5-style model"},
    "P4": {"title": "PatchTST", "year": 2023, "method": "patching + channel independence"},
    "P5": {"title": "iTransformer", "year": 2024, "method": "attention over variates"},
    "P6": {"title": "Moirai", "year": 2024, "method": "any-variate attention, universal forecaster"},
}
STATE = {"search_step": 0, "found": []}   # search_step: counted by the planning graph's executor (simulated data)

In [ ]:
@tool
def search_papers(query: str) -> str:
    """Search papers on a topic. Returns a JSON list of {id, title, year}."""
    ids = ["P1", "P2", "P3"] if STATE["search_step"] <= 1 else list(PAPERS)
    STATE["found"] = ids
    return json.dumps([{"id": i, "title": PAPERS[i]["title"], "year": PAPERS[i]["year"]} for i in ids])

@tool
def read_papers(paper_ids: list[str]) -> str:
    """Read papers by id (e.g. 'P1') and return their key methods."""
    unknown = [i for i in paper_ids if i not in PAPERS]
    if unknown:
        return f"Error: unknown ids {unknown}. Use ids from search_papers, e.g. 'P1'."
    return json.dumps([{"title": PAPERS[i]["title"], "method": PAPERS[i]["method"]} for i in paper_ids])

tools = [search_papers, read_papers]
print(read_papers.name, "|", read_papers.description)
print(read_papers.args)

## 2. State

The **State** is a dict that flows through every node — each node returns only the fields it wants to update
Fields with a **reducer** (e.g. `add_messages`) are *appended to*, while plain fields are *overwritten*

In [ ]:
# ✏️ TODO 1: The state has 2 fields: messages (a list of BaseMessage, appended via the add_messages reducer) and iteration (an int counting rounds)
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], ___]
    iteration: ___

## 3. Nodes

A node is a plain function: **takes the state → returns a dict of the updated parts**

In [ ]:
MAX_ITERATIONS = 6
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: AgentState):
    reply = llm_with_tools.invoke(state["messages"])
    return {"messages": [reply], "iteration": state["iteration"] + 1}

tool_node = ToolNode(tools)

def stop_node(state: AgentState):
    return {"messages": [AIMessage("⚠️ Stopped: reached max iterations")]}

## 4. Edges — who goes next?

A **conditional edge** is a function that looks at the state and answers *which node goes next* — this is ④ termination and ③ max iterations from Lab 1

In [ ]:
# ✏️ TODO 2: ④ termination: if the last message has no tool calls, the model is done, so go to END
def route(state: AgentState):
    last = state["messages"][-1]
    if ___:
        return ___
    if state["iteration"] >= MAX_ITERATIONS:
        return "stop"
    return "tools"

In [ ]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("stop", stop_node)

# ✏️ TODO 3: Connect the edges: start at agent → after agent, route decides → tools loops back to agent → stop ends at END
builder.add_edge(START, "agent")
builder.___("agent", route, ["tools", "stop", END])
builder.add_edge("tools", ___)
builder.add_edge("stop", END)

graph = builder.compile()
graph.get_graph().print_ascii()

## 5. Run the graph

`graph.stream(..., stream_mode="updates")` yields results one node at a time — so you see what each node did

In [ ]:
def show(update):
    for node, change in update.items():
        for m in change.get("messages", []):
            calls = getattr(m, "tool_calls", None)
            if calls:
                for c in calls:
                    print(f"🔧 [{node}] {c['name']}({c['args']})")
            else:
                print(f"{'👀' if node == 'tools' else '💬'} [{node}]", str(m.content)[:150])

In [ ]:
task = ("Find transformer models for time-series forecasting, read them, "
        "and explain in 3 bullet points how they differ.")
for update in graph.stream({"messages": [HumanMessage(task)], "iteration": 0}, stream_mode="updates"):
    show(update)

## 6. Checkpoint + human-in-the-loop

What Lab 1 **doesn't have**: a checkpointer saves the state after every node, so we can **pause for human approval** before running a tool, then **resume from the same point**

The textbook's example uses `SqliteSaver` (stores to a file) — this lab uses `InMemorySaver` (stores in memory) so it runs easily in Colab

In [ ]:
# ✏️ TODO 4: Compile again with a checkpointer, and tell it to pause before entering the tools node
safe_graph = builder.compile(checkpointer=___, interrupt_before=___)
config = {"configurable": {"thread_id": "lab5-demo"}}

In [ ]:
safe_graph.invoke({"messages": [HumanMessage("Find transformer models for time-series forecasting and name them.")],
                   "iteration": 0}, config)
paused = safe_graph.get_state(config)
print("⏸️ next node:", paused.next)
for c in paused.values["messages"][-1].tool_calls:
    print(f"   🔧 awaiting approval: {c['name']}({c['args']})")

In [ ]:
approved = True   # in real systems: wait for a human to approve
if approved:
    result = safe_graph.invoke(None, config)
    print("▶️ resumed →", result["messages"][-1].content[:300])
print("💾 checkpoints saved:", len(list(safe_graph.get_state_history(config))))

## 7. Planning graph — plan → execute → replan

Lab 2's planning agent can be written as a graph with 3 nodes — **the executor uses the ReAct graph from part 4 inside it** (a graph inside a graph)

```
START → planner → executor ──(steps left)──→ executor
                     │
                     ├──(search found too few)──→ replan → executor
                     └──(all steps done)──→ END
```

This graph has an edge that loops back into itself (executor → executor), which `print_ascii()` can't draw, so it's drawn above instead

In [ ]:
class Step(BaseModel):
    task: str
    tool: Literal["search_papers", "read_papers", "none"]

class Plan(BaseModel):
    steps: list[Step]

PLANNER_RULES = ("You are a planner. Return 3-4 steps. Each step uses exactly one tool:\n"
                 "- search_papers: find papers\n- read_papers: read papers found earlier\n"
                 "- none: write text (no tool)\nThe LAST step must write the final report (tool=none).\n")
planner = llm.with_structured_output(Plan)

In [ ]:
# ✏️ TODO 5: results must append each step's result (use the operator.add reducer), while steps can be overwritten on replan
class PlanState(TypedDict):
    request: str
    steps: list[Step]
    results: Annotated[list[str], ___]
    replans: int

def show_plan(steps, done=0):
    for i, s in enumerate(steps):
        print(f"  {'✅' if i < done else '⬜'} {i + 1}. {s.task[:90]}  (tool={s.tool})")

In [ ]:
def planner_node(state: PlanState):
    plan = planner.invoke(PLANNER_RULES + f"Request: {state['request']}")
    print("📋 PLAN"); show_plan(plan.steps)
    return {"steps": plan.steps}

In [ ]:
def executor_node(state: PlanState):
    step = state["steps"][len(state["results"])]
    print(f"⚙️ step {len(state['results']) + 1}: {step.task[:90]}")
    if step.tool == "search_papers":
        STATE["search_step"] += 1          # for the simulated database only
    task = (f"Overall goal: {state['request']}\nResults so far:\n" + "\n".join(state["results"]) +
            f"\nYour step: {step.task}\nDo only this step. Be concise. Keep paper ids (e.g. P1) in your output.")
    out = graph.invoke({"messages": [HumanMessage(task)], "iteration": 0})
    return {"results": [out["messages"][-1].content]}

In [ ]:
# ✏️ TODO 6: Replan when the step just done was a search but found fewer than MIN_PAPERS papers (and at most 2 replans)
MIN_PAPERS = 5

def check(state: PlanState):
    step = state["steps"][len(state["results"]) - 1]
    if step.tool == "search_papers" and ___ and state["replans"] < 2:
        print(f"   ⚠️ found only {len(STATE['found'])} papers (need {MIN_PAPERS}) → REPLAN")
        return "replan"
    return "executor" if len(state["results"]) < len(state["steps"]) else END

In [ ]:
def replan_node(state: PlanState):
    done = len(state["results"])
    history = "\n".join(f"- {s.task}: {r[:200]}" for s, r in zip(state["steps"], state["results"]))
    plan = planner.invoke(PLANNER_RULES + f"Request: {state['request']}\nCompleted steps and results:\n{history}\n"
                          f"Problem: the search found only {len(STATE['found'])} papers; the report needs at least {MIN_PAPERS}.\n"
                          "Return ONLY the remaining steps (not the completed ones): "
                          "a broader search first, then reading, then the final report.")
    steps = state["steps"][:done] + plan.steps
    print("📋 REVISED PLAN"); show_plan(steps, done)
    return {"steps": steps, "replans": state["replans"] + 1}

In [ ]:
pb = StateGraph(PlanState)
pb.add_node("planner", planner_node)
pb.add_node("executor", executor_node)
pb.add_node("replan", replan_node)

pb.add_edge(START, "planner")
pb.add_edge("planner", "executor")
pb.add_conditional_edges("executor", check, ["executor", "replan", END])
pb.add_edge("replan", "executor")
plan_graph = pb.compile()

In [ ]:
STATE.update(search_step=0, found=[])
final = plan_graph.invoke({"request": "Write a short report comparing transformer architectures for time-series forecasting.",
                           "steps": [], "results": [], "replans": 0})
print("\n📄 REPORT:\n", final["results"][-1][:1000])

## Summary

| Written ourselves (Labs 1–2) | LangGraph (Lab 5) |
|---|---|
| `messages.append(...)` | State + reducer `add_messages` |
| `ask_llm(...)` + `run_tool(...)` | node `agent` + `ToolNode` |
| `for i in range(max_iterations)` | edge `tools → agent` + `iteration` |
| `if not calls: return` | conditional edge `route` → `END` |
| `execute_plan()` while loop + `replan()` | node `planner / executor / replan` + conditional edge `check` |
| `PLAN_FORMAT` (hand-written JSON schema) | `with_structured_output(Plan)` |
| — | checkpointer + `interrupt_before` (human-in-the-loop) |

**When to write it yourself / when to use a framework** — write it yourself to understand the mechanism and while the agent is still small;
use a framework when you need checkpoints, human approval, debugging, and a structure the whole team can read

**Try next:** compile the planning graph with a checkpointer and `interrupt_before=["executor"]` so a human approves the plan before it starts